<a href="https://colab.research.google.com/github/Abraham-Na70/B401-Cenim/blob/main/module/module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-on Module: Human-Robot Interaction Workshop
**B401 Laboratory: Embedded Systems and Robotics**

Welcome to the Human-Robot Interaction (HRI) Workshop. In this integrated hands-on module, you will learn how to bridge human perception (body movements, hand gestures, and natural language instructions) with the physical actuation of modern robotic systems:
1. **Industrial Robot Manipulator (Universal Robots UR5)** for high-precision manipulation tasks (Pick and Place).
2. **Quadruped Mobile Robot (RoboDog)** for dynamic navigation and environmental mobility tasks.

---

### Workshop Workflow:
* **Chapter 0: Environment Setup and Dependency Installation**
  Setting up the Colab runtime environment, installing Computer Vision, Machine Learning, and WebSockets libraries, and downloading the MediaPipe Hand Landmarker model.
* **Chapter 1: Visual Perception and Hand Gesture Recognition (Computer Vision & Machine Learning)**
  Extracting 21 hand joint coordinates via MediaPipe, invariant 42-feature spatial normalization, interactive dataset recording studio, and Random Forest classification.
* **Chapter 2: Complex Robot Commands Using Large Language Models (LLM)**
  Translating natural language instructions into structured action plans using prompt engineering and LLM reasoning.
* **Chapter 3: Hands-on Learning with the UR5 Robotic Manipulator**
  Distributed network architecture, Time-Leased Mutex concurrency management, chessboard workspace coordinates, dwell stabilization filter, and live physical robot teleoperation.
* **Chapter 4: Interaction and Control of Quadruped Robots (RoboDog)**
  Robot API connection, body pose mapping to posture/stance modes, and dynamic interaction scenarios.

# Chapter 0: Environment Setup and Dependency Installation

## 0.1 Conceptual Context: Self-Contained Computing Environment

Before diving into the Human-Robot Interaction (HRI) workshop, we need to prepare our Google Colab working environment. In this chapter, we install all required software dependencies—spanning Computer Vision libraries (MediaPipe, OpenCV), Machine Learning tools (Scikit-Learn), and asynchronous communication protocols (WebSockets).

Additionally, we download the pre-trained MediaPipe hand landmarker model weights (`hand_landmarker.task`) locally into the Colab environment. This ensures that the hand pose recognition and gesture training in **Chapter 1** can run entirely self-contained (offline) inside your browser without requiring a live connection to the lab's robot server.

In [ ]:
#@title 0.1 Library Dependencies Installation and Hand Detection Model Download { display-mode: "form" }
# 0.1 Library Dependencies Installation and Hand Detection Model Download
# Please do not modify the contents of this cell

import os
import sys

print("[1/2] Checking and installing required libraries...")
!pip install -q websockets mediapipe opencv-python-headless scikit-learn matplotlib joblib openai

MODEL_PATH = "hand_landmarker.task"
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"

print("[2/2] Checking MediaPipe Hand Landmarker model availability...")
if not os.path.exists(MODEL_PATH):
    print(f"Downloading model from {MODEL_URL}...")
    !wget -q -O {MODEL_PATH} {MODEL_URL}
    print("Model successfully downloaded.")
else:
    print("Model file is already available in the working directory.")

print("Installation complete. Environment is ready to use.")


# Chapter 1: Visual Perception and Hand Gesture Recognition (Computer Vision & Machine Learning)

## 1.1 Reading Hand Finger Skeletons with MediaPipe

> **Quick Goal:** Convert raw camera video into 21 finger joint coordinates so the computer recognizes gesture shapes without being distracted by clothing colors or background clutter.

* **The Photo vs. Stick Figure Analogy:**
  * When guessing whether a friend is giving a thumbs-up or waving, we only focus on the **positions and bends of their finger bones**.
  * We don't care about the color of their shirt, the wall paint in the room, or their skin tone.
  * A laptop camera captures millions of raw color pixels, which overwhelms computers and slows them down. We simplify this into **21 finger joint coordinates**, just like drawing a stick figure.
* **The Automatic Magnifying Glass Analogy (Normalization):**
  * When your hand moves closer to the camera, it appears huge. When you pull it back, it shrinks.
  * To keep the AI from getting confused, the hand skeleton is automatically scaled to a uniform, standard size.
  * The wrist point is anchored as the origin benchmark `(0, 0)`.

### How It Works Behind the Scenes
* The MediaPipe algorithm detects 21 coordinates `(x, y)` of finger joints from the camera frame.
* Each joint coordinate is offset relative to the wrist position, then scaled by the palm length.
* The result is a consistent 42-number decimal vector, regardless of whether your hand is on the left, right, close, or far from the camera.

> **Processing Pipeline:**  
> `Raw Camera Frame (900,000+ Pixels)` ➔ `21 Finger Joint Coordinates (MediaPipe)` ➔ `Automatic Scale Normalization` ➔ `42 Standard Invariant Numbers (AI-Ready)`

> **Class Discussion Question:** *If you wore thick winter gloves or sat in a dimly lit room, would this stick figure model still be able to detect finger joints? Why or why not?*

In [ ]:
#@title 1.1 Initialize Hand Landmark Detector (Universal MediaPipe) { display-mode: "form" }
# Please do not modify the contents of this cell

import os
import cv2
import numpy as np
import mediapipe as mp

class HandResult:
    """Wrapper to unify output structures across modern and legacy MediaPipe."""
    def __init__(self, landmarks):
        self.multi_hand_landmarks = [landmarks] if landmarks is not None else None

class UniversalHandDetector:
    """
    Flexible hand detector:
    Supports modern MediaPipe (>=0.10.30, Tasks API) and legacy (mp.solutions.hands).
    """
    def __init__(self, model_path: str = "hand_landmarker.task"):
        self.mode = "legacy"
        if hasattr(mp, "solutions") and hasattr(mp.solutions, "hands"):
            self.detector = mp.solutions.hands.Hands(
                static_image_mode=True,
                max_num_hands=1,
                min_detection_confidence=0.6,
                min_tracking_confidence=0.6,
            )
        else:
            self.mode = "tasks"
            if not os.path.exists(model_path):
                import urllib.request
                url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
                print(f"Downloading MediaPipe Tasks model ({model_path})...")
                urllib.request.urlretrieve(url, model_path)

            from mediapipe.tasks import python
            from mediapipe.tasks.python import vision

            base_options = python.BaseOptions(model_asset_path=model_path)
            options = vision.HandLandmarkerOptions(
                base_options=base_options,
                running_mode=vision.RunningMode.IMAGE,
                num_hands=1,
                min_hand_detection_confidence=0.6,
                min_tracking_confidence=0.6,
            )
            self.detector = vision.HandLandmarker.create_from_options(options)

    def detect_landmarks(self, rgb_image: np.ndarray):
        """Accepts RGB image (H, W, 3), returns 21 hand landmark coordinates."""
        if self.mode == "legacy":
            results = self.detector.process(rgb_image)
            if results.multi_hand_landmarks:
                return results.multi_hand_landmarks[0]
            return None
        else:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
            results = self.detector.detect(mp_image)
            if results.hand_landmarks and len(results.hand_landmarks) > 0:
                return results.hand_landmarks[0]
            return None

    def process(self, rgb_image: np.ndarray):
        """Interface method aligned with mp.solutions.hands.process()."""
        lms = self.detect_landmarks(rgb_image)
        return HandResult(lms)


hands_detector = UniversalHandDetector()

GESTURE_LABELS = [
    "POINT",
    "PEACE",
    "THREE_FINGERS",
    "FOUR_FINGERS",
    "OK_SIGN",
    "OPEN_PALM",
    "THUMBS_UP",
    "FIST",
]

def extract_normalized_features(landmarks) -> np.ndarray:
    """
    Converts 21 MediaPipe landmark points into a 42-feature spatial invariant vector:
      1. Translation: shifts wrist (point 0) to (0, 0).
      2. Scale: divides coordinates by distance from wrist to middle finger MCP (point 9).
    """
    if landmarks is None:
        return None
    lms = getattr(landmarks, "landmark", landmarks)
    coords = np.array([[lm.x, lm.y] for lm in lms])
    if len(coords) < 21:
        return None

    wrist = coords[0]
    translated = coords - wrist

    palm_span = np.linalg.norm(translated[9])
    if palm_span < 1e-6:
        palm_span = 1.0
    normalized = translated / palm_span

    return normalized.flatten()

print(f"Universal Hand Detector ({hands_detector.mode.upper()} mode) dan fungsi ekstraksi 42-fitur siap.")


## 1.2 Collecting a Hand Gesture Sample Dataset

> **Quick Goal:** Record custom hand pose samples directly via your laptop webcam to train an AI model to recognize your specific gesture vocabulary.

* **8 Robot Control Commands:**
  * `POINT` (1 index finger) : Navigate to Quadrant 1.
  * `PEACE` (2 fingers: index & middle) : Navigate to Quadrant 2.
  * `THREE_FINGERS` (3 fingers) : Navigate to Quadrant 3.
  * `FOUR_FINGERS` (4 fingers) : Navigate to Quadrant 4.
  * `OK_SIGN` (thumb-index circle) : Grip / Pick up an object (*Pick Sequence*).
  * `OPEN_PALM` (open palm) : Release gripper / Place object into box (*Release Bin*).
  * `THUMBS_UP` (thumbs up) : Return to neutral ready position (*Home / Ready*).
  * `FIST` (clenched fist) : Idle / Hold position (*Hold / Neutral*).
* **Tips for High-Quality Recording:**
  * Click your desired gesture button, watch the 3-second countdown, and hold your hand steady in front of the camera.
  * Slightly rotate or tilt your hand as the camera captures 10 rapid burst frames. This teaches the AI that a slightly tilted thumb is still a thumb!
  * If your laptop webcam has issues, use the **Load Synthetic Baseline** button to populate standard training samples automatically.

### How It Works Behind the Scenes
* The laptop camera streams via the browser (JavaScript WebRTC Canvas) and transfers frames into Google Colab's Python environment.
* Each captured frame immediately extracts 21 joint coordinates and saves them neatly into our dataset table.

> **Dataset Recording Pipeline:**  
> `Select Gesture Button` ➔ `Countdown 3.. 2.. 1..` ➔ `Rapid 10-Frame Burst Capture` ➔ `21 Joint Landmark Extraction` ➔ `Training-Ready Dataset`

> **Class Discussion Question:** *Why is it recommended to gently tilt and shift your hand while recording, rather than holding it completely still like a statue?*

In [ ]:
#@title 1.2 Gesture Dataset Recording Studio (Browser Webcam) { display-mode: "form" }
# Please do not modify the contents of this cell
# 1.2 Gesture Dataset Collection: Interactive 8-Gesture Grid Studio
import time
import base64
import numpy as np
import cv2
from google.colab import output
from IPython.display import JSON, Javascript, display

DATASET_X = []
DATASET_Y = []

GESTURE_CLASSES = [
    ("POINT", "POINT (1 Finger)"),
    ("PEACE", "PEACE (2 Fingers)"),
    ("THREE_FINGERS", "THREE (3 Fingers)"),
    ("FOUR_FINGERS", "FOUR (4 Fingers)"),
    ("OK_SIGN", "OK_SIGN (Pick)"),
    ("OPEN_PALM", "OPEN_PALM (Release)"),
    ("THUMBS_UP", "THUMBS_UP (Home)"),
    ("FIST", "FIST (Neutral/Hold)"),
]

# ── A. Python Callback: Process Burst Frames in Replace/Append Mode ──
def process_burst_payload(gesture_label, frames_b64, mode="replace"):
    """
    Receives frames from JS.
    mode='replace': Clears previous recordings for this gesture before appending new ones.
    mode='append': Appends to existing recordings.
    """
    global DATASET_X, DATASET_Y

    if mode == "replace":
        filtered_x = [x for x, y in zip(DATASET_X, DATASET_Y) if y != gesture_label]
        filtered_y = [y for y in DATASET_Y if y != gesture_label]
        DATASET_X = filtered_x
        DATASET_Y = filtered_y

    valid_count = 0
    annotated_thumbnails = []
    
    # 21 MediaPipe hand joint landmark connections
    HAND_CONNECTIONS = [
        (0, 1), (1, 2), (2, 3), (3, 4),        # Jempol (Thumb)
        (0, 5), (5, 6), (6, 7), (7, 8),        # Telunjuk (Index)
        (5, 9), (9, 10), (10, 11), (11, 12),   # Jari Tengah (Middle)
        (9, 13), (13, 14), (14, 15), (15, 16), # Jari Manis (Ring)
        (13, 17), (17, 18), (18, 19), (19, 20),# Kelingking (Pinky)
        (0, 17)                                # Dasar Telapak (Palm)
    ]

    for b64_str in frames_b64:
        try:
            header, encoded = b64_str.split(',', 1) if ',' in b64_str else ('', b64_str)
            img_bytes = base64.b64decode(encoded)
            img_np = np.frombuffer(img_bytes, dtype=np.uint8)
            frame = cv2.imdecode(img_np, flags=cv2.IMREAD_COLOR)
            if frame is None:
                continue
            flipped = cv2.flip(frame, 1)
            rgb = cv2.cvtColor(flipped, cv2.COLOR_BGR2RGB)

            lms = hands_detector.detect_landmarks(rgb)
            if lms is not None:
                feats = extract_normalized_features(lms)
                if feats is not None:
                    DATASET_X.append(feats)
                    DATASET_Y.append(gesture_label)
                    valid_count += 1
                    
                    # Save skeleton visualization preview & 21 joints for the first 3 samples
                    if len(annotated_thumbnails) < 3:
                        h, w, _ = flipped.shape
                        annotated = flipped.copy()
                        for s_idx, e_idx in HAND_CONNECTIONS:
                            if s_idx < len(lms) and e_idx < len(lms):
                                p1 = (int(lms[s_idx][0] * w), int(lms[s_idx][1] * h))
                                p2 = (int(lms[e_idx][0] * w), int(lms[e_idx][1] * h))
                                cv2.line(annotated, p1, p2, (0, 255, 120), 2, cv2.LINE_AA)
                        for pt_idx, pt in enumerate(lms):
                            cx, cy = int(pt[0] * w), int(pt[1] * h)
                            color = (0, 0, 255) if pt_idx == 0 else (0, 255, 255) if pt_idx in (4, 8, 12, 16, 20) else (255, 120, 0)
                            cv2.circle(annotated, (cx, cy), 4, color, -1, cv2.LINE_AA)
                            cv2.circle(annotated, (cx, cy), 5, (255, 255, 255), 1, cv2.LINE_AA)
                        
                        thumb = cv2.resize(annotated, (140, 105))
                        _, buf = cv2.imencode('.jpg', thumb, [cv2.IMWRITE_JPEG_QUALITY, 75])
                        thumb_b64 = "data:image/jpeg;base64," + base64.b64encode(buf.tobytes()).decode('ascii')
                        annotated_thumbnails.append(thumb_b64)
        except Exception:
            continue

    counts = {g[0]: DATASET_Y.count(g[0]) for g in GESTURE_CLASSES}
    total = len(DATASET_Y)
    return JSON({"valid": valid_count, "total": total, "counts": counts, "thumbnails": annotated_thumbnails})

output.register_callback("notebook.ProcessBurstFrames", process_burst_payload)

# ── B. Reset Dataset Callback ──
def py_reset_dataset():
    global DATASET_X, DATASET_Y
    DATASET_X.clear()
    DATASET_Y.clear()
    counts = {g[0]: 0 for g in GESTURE_CLASSES}
    return JSON({"total": 0, "counts": counts})

output.register_callback("notebook.ResetDataset", py_reset_dataset)

# ── C. Fallback: Generator Data Sintetis ──
def py_load_synthetic():
    global DATASET_X, DATASET_Y
    DATASET_X.clear()
    DATASET_Y.clear()

    templates = {
        "FIST": [[0, 0]] + [[0.05 * i, -0.2] for i in range(1, 5)] + [[0.0, -0.35], [0.0, -0.3], [0.0, -0.25], [0.0, -0.2]] * 4,
        "POINT": [[0, 0]] + [[-0.1, -0.1], [-0.15, -0.15], [-0.18, -0.2], [-0.2, -0.25]] + [[0.0, -0.4], [0.0, -0.65], [0.0, -0.85], [0.0, -1.05]] + [[0.1, -0.35], [0.1, -0.3], [0.1, -0.25], [0.1, -0.2]] * 3,
        "PEACE": [[0, 0]] + [[-0.1, -0.1], [-0.15, -0.15], [-0.18, -0.2], [-0.2, -0.25]] + [[-0.08, -0.4], [-0.12, -0.65], [-0.15, -0.85], [-0.18, -1.05]] + [[0.08, -0.4], [0.12, -0.65], [0.15, -0.85], [0.18, -1.05]] + [[0.1, -0.35], [0.1, -0.3], [0.1, -0.25], [0.1, -0.2]] * 2,
        "THREE_FINGERS": [[0, 0]] + [[-0.1, -0.1], [-0.15, -0.15], [-0.18, -0.2], [-0.2, -0.25]] + [[-0.15, -0.4], [-0.2, -0.65], [-0.25, -0.85], [-0.3, -1.05]] + [[0.0, -0.4], [0.0, -0.65], [0.0, -0.85], [0.0, -1.05]] + [[0.15, -0.4], [0.2, -0.65], [0.25, -0.85], [0.3, -1.05]] + [[0.2, -0.35], [0.2, -0.3], [0.2, -0.25], [0.2, -0.2]],
        "FOUR_FINGERS": [[0, 0]] + [[-0.05, -0.1], [-0.08, -0.15], [-0.1, -0.2], [-0.12, -0.22]] + [[-0.2, -0.4], [-0.25, -0.65], [-0.3, -0.85], [-0.35, -1.05]] + [[-0.07, -0.4], [-0.09, -0.65], [-0.1, -0.85], [-0.12, -1.05]] + [[0.07, -0.4], [0.09, -0.65], [0.1, -0.85], [0.12, -1.05]] + [[0.2, -0.4], [0.25, -0.65], [0.3, -0.85], [0.35, -1.05]],
        "OPEN_PALM": [[0, 0]] + [[-0.25, -0.2], [-0.35, -0.3], [-0.45, -0.4], [-0.55, -0.5]] + [[-0.2, -0.4], [-0.25, -0.65], [-0.3, -0.85], [-0.35, -1.05]] + [[-0.07, -0.4], [-0.09, -0.65], [-0.1, -0.85], [-0.12, -1.05]] + [[0.07, -0.4], [0.09, -0.65], [0.1, -0.85], [0.12, -1.05]] + [[0.2, -0.4], [0.25, -0.65], [0.3, -0.85], [0.35, -1.05]],
        "THUMBS_UP": [[0, 0]] + [[-0.1, -0.3], [-0.15, -0.55], [-0.2, -0.8], [-0.25, -1.05]] + [[0.05, -0.25], [0.05, -0.2], [0.05, -0.15], [0.05, -0.1]] * 4,
        "OK_SIGN": [[0, 0]] + [[-0.1, -0.2], [-0.12, -0.3], [-0.1, -0.4], [-0.05, -0.45]] + [[0.0, -0.2], [0.0, -0.3], [0.0, -0.4], [-0.05, -0.45]] + [[0.1, -0.4], [0.12, -0.65], [0.15, -0.85], [0.18, -1.05]] + [[0.2, -0.4], [0.22, -0.65], [0.25, -0.85], [0.28, -1.05]] + [[0.3, -0.35], [0.32, -0.55], [0.35, -0.75], [0.38, -0.95]],
    }

    np.random.seed(42)
    for label, base_coords in templates.items():
        base = np.array(base_coords[:21])
        for _ in range(40):
            noise = np.random.normal(0, 0.025, base.shape)
            jittered = base + noise
            palm_span = np.linalg.norm(jittered[9]) or 1.0
            feats = (jittered / palm_span).flatten()
            DATASET_X.append(feats)
            DATASET_Y.append(label)

    counts = {g[0]: DATASET_Y.count(g[0]) for g in GESTURE_CLASSES}
    return JSON({"total": len(DATASET_Y), "counts": counts})

output.register_callback("notebook.LoadSyntheticBaseline", py_load_synthetic)

# ── D. Dashboard UI Terpercaya: Auto Hardware Track Release & Resilient Perms ──
DASHBOARD_JS = """
async function initDatasetStudio() {
  // 1. Release ANY orphaned camera stream from prior runs
  if (window.activeDatasetStream) {
    try { window.activeDatasetStream.getTracks().forEach(t => t.stop()); } catch(e){}
    window.activeDatasetStream = null;
  }
  if (window.teleopStream) {
    try { window.teleopStream.getTracks().forEach(t => t.stop()); } catch(e){}
    window.teleopStream = null;
  }

  const existing = document.getElementById('gesture-dashboard-box');
  if (existing) existing.remove();

  const container = document.createElement('div');
  container.id = 'gesture-dashboard-box';
  container.style.fontFamily = '-apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif';
  container.style.background = '#1e1e1e';
  container.style.color = '#fff';
  container.style.padding = '18px';
  container.style.borderRadius = '12px';
  container.style.maxWidth = '680px';
  container.style.boxShadow = '0 4px 14px rgba(0,0,0,0.5)';
  container.style.marginTop = '10px';

  container.innerHTML = `
    <div style="display:flex; justify-content:space-between; align-items:center; border-bottom: 1px solid #333; padding-bottom: 10px; margin-bottom: 12px;">
      <div>
        <h3 style="margin:0; font-size: 18px; color:#00ffcc;">Hand Gesture Dataset Collection Studio</h3>
        <div style="font-size: 12px; color: #aaa; margin-top:3px;">Pilih gesture -> hitung mundur 3s -> rekam 10 frame burst.</div>
      </div>
      <div id="dash-total-badge" style="background:#0984e3; padding: 4px 10px; border-radius: 6px; font-weight:bold; font-size:13px;">Total: 0 Sampel</div>
    </div>

    <!-- Mode Selector & Controls -->
    <div style="display:flex; justify-content:space-between; align-items:center; background:#2a2a2a; padding:6px 12px; border-radius:6px; margin-bottom:12px; font-size:12px;">
      <div style="display:flex; align-items:center; gap:12px;">
        <span>Mode Rekam:</span>
        <label style="cursor:pointer;"><input type="radio" name="recMode" value="replace" checked> Ganti Ulang (Replace)</label>
        <label style="cursor:pointer;"><input type="radio" name="recMode" value="append"> Tambah (Append)</label>
      </div>
      <button id="btn-stop-cam" style="background:#c0392b; color:#fff; border:none; padding:4px 10px; border-radius:4px; font-size:11px; cursor:pointer; font-weight:bold;">Turn Off Camera</button>
    </div>

    <div style="display: flex; gap: 16px; margin-bottom: 14px;">
      <div style="position: relative; width: 280px; height: 210px; background: #000; border-radius: 8px; overflow: hidden; flex-shrink: 0; display:flex; align-items:center; justify-content:center;">
        <video id="dash-video" style="width: 100%; height: 100%; object-fit: cover; transform: scaleX(-1); display:none;" autoplay muted playsinline></video>
        <div id="dash-placeholder" style="color: #888; font-size: 12px; text-align: center; padding: 10px;">Camera standby.<br>Click a gesture button to begin.</div>
        <div id="dash-countdown" style="display:none; position: absolute; font-size: 54px; font-weight: bold; color: #f1c40f; text-shadow: 0 2px 8px rgba(0,0,0,0.8);">3</div>
        <div id="dash-rec-bar" style="display:none; position: absolute; bottom: 8px; left: 8px; right: 8px; background: rgba(231,76,60,0.9); color:#fff; font-size:11px; padding: 4px; text-align:center; border-radius: 4px; font-weight:bold;">Merekam...</div>
      </div>

      <div style="flex-grow: 1; display: grid; grid-template-columns: 1fr 1fr; gap: 6px; align-content: start;">
        <div class="stat-badge" id="b-POINT" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">POINT: <b>0</b></div>
        <div class="stat-badge" id="b-PEACE" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">PEACE: <b>0</b></div>
        <div class="stat-badge" id="b-THREE_FINGERS" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">THREE: <b>0</b></div>
        <div class="stat-badge" id="b-FOUR_FINGERS" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">FOUR: <b>0</b></div>
        <div class="stat-badge" id="b-OK_SIGN" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">OK_SIGN: <b>0</b></div>
        <div class="stat-badge" id="b-OPEN_PALM" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">PALM: <b>0</b></div>
        <div class="stat-badge" id="b-THUMBS_UP" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">THUMBS: <b>0</b></div>
        <div class="stat-badge" id="b-FIST" style="background:#2d3436; padding:6px 8px; border-radius:6px; font-size:11px;">FIST: <b>0</b></div>
      </div>
    </div>

    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 8px; margin-bottom: 12px;">
      <button class="rec-btn" id="btn-POINT">POINT</button>
      <button class="rec-btn" id="btn-PEACE">PEACE</button>
      <button class="rec-btn" id="btn-THREE_FINGERS">THREE</button>
      <button class="rec-btn" id="btn-FOUR_FINGERS">FOUR</button>
      <button class="rec-btn" id="btn-OK_SIGN">OK_SIGN</button>
      <button class="rec-btn" id="btn-OPEN_PALM">PALM</button>
      <button class="rec-btn" id="btn-THUMBS_UP">THUMBS</button>
      <button class="rec-btn" id="btn-FIST">FIST</button>
    </div>

    <div style="display:flex; justify-content:space-between; align-items:center; border-top:1px solid #333; padding-top:10px;">
      <div style="display:flex; gap:8px;">
        <button id="btn-synth" style="background:#34495e; color:#ecf0f1; border:none; padding:6px 12px; border-radius:5px; font-size:11px; cursor:pointer;">Load Synthetic Baseline</button>
        <button id="btn-reset" style="background:#7f8c8d; color:#ecf0f1; border:none; padding:6px 12px; border-radius:5px; font-size:11px; cursor:pointer;">Reset Data</button>
      </div>
      <div id="dash-info" style="font-size:12px; color:#2ecc71;">Siap merekam data.</div>
    </div>
  `;

  document.querySelector('#output-body').appendChild(container);

  const style = document.createElement('style');
  style.textContent = `
    .rec-btn { background: #27ae60; color: white; border: none; padding: 8px 4px; border-radius: 6px; font-size: 11px; font-weight: bold; cursor: pointer; transition: 0.2s; }
    .rec-btn:hover { background: #2ecc71; }
    .rec-btn:disabled { background: #555; cursor: not-allowed; }
  `;
  container.appendChild(style);

  let isRecording = false;

  function stopCamera() {
    if (window.activeDatasetStream) {
      window.activeDatasetStream.getTracks().forEach(t => t.stop());
      window.activeDatasetStream = null;
    }
    const video = document.getElementById('dash-video');
    const placeholder = document.getElementById('dash-placeholder');
    if (video) video.style.display = 'none';
    if (placeholder) {
      placeholder.style.display = 'block';
      placeholder.innerHTML = 'Camera disabled.<br>Click a gesture button to reactivate.';
    }
    const info = document.getElementById('dash-info');
    if (info) { info.innerText = 'Camera turned off.'; info.style.color = '#e74c3c'; }
  }

  function updateBadgeCounts(counts, total) {
    document.getElementById('dash-total-badge').innerText = `Total: ${total} Sampel`;
    for (const [key, val] of Object.entries(counts)) {
      const el = document.getElementById(`b-${key}`);
      if (el) {
        const icon = el.innerText.split(' ')[0];
        el.innerHTML = `${icon} ${key.split('_')[0]}: <b>${val}</b>`;
        if (val >= 10) el.style.background = '#1e8449';
        else if (val > 0) el.style.background = '#d35400';
        else el.style.background = '#2d3436';
      }
    }
  }

  async function getCameraStream() {
    if (window.activeDatasetStream && window.activeDatasetStream.active) {
      return window.activeDatasetStream;
    }
    const stream = await navigator.mediaDevices.getUserMedia({
      video: { width: { ideal: 480 }, height: { ideal: 360 }, facingMode: "user" }
    });
    window.activeDatasetStream = stream;
    return stream;
  }

  async function triggerBurst(label) {
    if (isRecording) return;
    isRecording = true;
    container.querySelectorAll('.rec-btn').forEach(b => b.disabled = true);

    const video = document.getElementById('dash-video');
    const placeholder = document.getElementById('dash-placeholder');
    const cd = document.getElementById('dash-countdown');
    const recBar = document.getElementById('dash-rec-bar');
    const info = document.getElementById('dash-info');

    const mode = document.querySelector('input[name="recMode"]:checked')?.value || 'replace';

    placeholder.style.display = 'none';
    video.style.display = 'block';
    info.innerText = `Initializing camera for [${label}]...`;
    info.style.color = '#f1c40f';

    try {
      const stream = await getCameraStream();
      video.srcObject = stream;
      await video.play();

      cd.style.display = 'block';
      for (let c = 3; c > 0; c--) {
        cd.innerText = c;
        await new Promise(r => setTimeout(r, 1000));
      }
      cd.style.display = 'none';

      recBar.style.display = 'block';
      info.innerText = `Recording [${label}]! Slightly tilt/shift your hand...`;
      info.style.color = '#e74c3c';

      const canvas = document.createElement('canvas');
      canvas.width = 400;
      canvas.height = 300;
      const ctx = canvas.getContext('2d');
      const frames = [];

      for (let i = 0; i < 10; i++) {
        ctx.drawImage(video, 0, 0, 400, 300);
        frames.push(canvas.toDataURL('image/jpeg', 0.65));
        recBar.innerText = `Merekam Frame ${i+1}/10...`;
        await new Promise(r => setTimeout(r, 200));
      }

      recBar.style.display = 'none';
      info.innerText = `Mengekstrak MediaPipe untuk [${label}]...`;
      info.style.color = '#3498db';

      const res = await google.colab.kernel.invokeFunction('notebook.ProcessBurstFrames', [label, frames, mode], {});
      
      let data = null;
      if (res && res.data) {
        if (res.data['application/json']) {
          data = res.data['application/json'];
        } else if (res.data['text/plain']) {
          try {
            data = JSON.parse(res.data['text/plain']);
          } catch(e) {
            try { data = JSON.parse(res.data['text/plain'].replace(/'/g, '"')); } catch(e2){}
          }
        }
      }

      if (data && data.counts) {
        updateBadgeCounts(data.counts, data.total);
        const actionWord = mode === 'replace' ? 'Ganti ulang' : 'Tambah';
        info.innerText = `[${label}]: ${actionWord} successful! (+${data.valid} valid frames).`;
        info.style.color = '#2ecc71';
      } else {
        info.innerText = `Successfully processed (data stored in Python).`;
        info.style.color = '#f1c40f';
      }

      if (data && data.thumbnails && data.thumbnails.length > 0) {
        const box = document.getElementById('dash-joint-box');
        const thumbs = document.getElementById('dash-joint-thumbs');
        if (box && thumbs) {
          box.style.display = 'block';
          thumbs.innerHTML = '';
          data.thumbnails.forEach((src, idx) => {
            const wrap = document.createElement('div');
            wrap.style.textAlign = 'center';
            const img = document.createElement('img');
            img.src = src;
            img.style.borderRadius = '4px';
            img.style.border = '1px solid #00ff78';
            img.style.width = '120px';
            img.style.height = '90px';
            const lbl = document.createElement('div');
            lbl.style.fontSize = '10px';
            lbl.style.color = '#00ff78';
            lbl.style.marginTop = '2px';
            lbl.innerText = `Sample ${idx + 1} (21 Joints)`;
            wrap.appendChild(img);
            wrap.appendChild(lbl);
            thumbs.appendChild(wrap);
          });
        }
      }

    } catch (err) {
      info.innerText = `Camera Error: ${err.message}`;
      info.style.color = '#e74c3c';
    } finally {
      isRecording = false;
      container.querySelectorAll('.rec-btn').forEach(b => b.disabled = false);
    }
  }

  // Bind Buttons
  const gestures = ["POINT", "PEACE", "THREE_FINGERS", "FOUR_FINGERS", "OK_SIGN", "OPEN_PALM", "THUMBS_UP", "FIST"];
  gestures.forEach(g => {
    const btn = document.getElementById(`btn-${g}`);
    if (btn) btn.onclick = () => triggerBurst(g);
  });

  document.getElementById('btn-stop-cam').onclick = stopCamera;

  document.getElementById('btn-synth').onclick = async () => {
    const info = document.getElementById('dash-info');
    info.innerText = "Memuat baseline...";
    const res = await google.colab.kernel.invokeFunction('notebook.LoadSyntheticBaseline', [], {});
    let data = res.data['application/json'] || JSON.parse(res.data['text/plain']?.replace(/'/g, '"') || '{}');
    if (data.counts) updateBadgeCounts(data.counts, data.total);
    info.innerText = `Baseline aktif (${data.total || 320} sampel)!`;
    info.style.color = '#2ecc71';
  };

  document.getElementById('btn-reset').onclick = async () => {
    const info = document.getElementById('dash-info');
    const res = await google.colab.kernel.invokeFunction('notebook.ResetDataset', [], {});
    let data = res.data['application/json'] || JSON.parse(res.data['text/plain']?.replace(/'/g, '"') || '{}');
    if (data.counts) updateBadgeCounts(data.counts, 0);
    info.innerText = "Dataset has been cleared.";
    info.style.color = '#e67e22';
  };
}
initDatasetStudio();
"""

display(Javascript(DASHBOARD_JS))


## 1.2.1 Visual Inspection of 21-Joint Hand Skeletons (Dataset Inspection)

> **Quick Goal:** Visualize the 21-joint stick figure skeleton representations from your recorded dataset before training the Random Forest model.

* **Validating Feature Extraction:**
  * The computer does not see your face or skin tone; it only perceives **joint coordinates and connecting finger bones** as illustrated below.
  * The red dot at the bottom is the wrist anchor `(0, 0)`.
  * Notice how the index finger stands erect for `POINT`, two fingers separate for `PEACE`, and all five fingers curl inward for `FIST`.
* **Dataset Balance:**
  * The title of each box displays the number of recorded samples (`N`).
  * Ensure each gesture has a balanced sample count (e.g., at least 10–40 samples per gesture) so the AI learns fairly across all classes without bias.

In [ ]:
#@title 1.2.1 Visualize 21 Hand Skeletons in Dataset { display-mode: "form" }
# Menampilkan visualisasi kerangka 2D untuk setiap kelas gestur yang ada di dataset

import matplotlib.pyplot as plt
import numpy as np

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),        # Jempol (Thumb)
    (0, 5), (5, 6), (6, 7), (7, 8),        # Telunjuk (Index)
    (0, 9), (9, 10), (10, 11), (11, 12),   # Jari Tengah (Middle)
    (0, 13), (13, 14), (14, 15), (15, 16), # Jari Manis (Ring)
    (0, 17), (17, 18), (18, 19), (19, 20), # Kelingking (Pinky)
    (5, 9), (9, 13), (13, 17), (0, 17)     # Palm
]

GESTURE_ORDER = [
    "POINT", "PEACE", "THREE_FINGERS", "FOUR_FINGERS",
    "OK_SIGN", "OPEN_PALM", "THUMBS_UP", "FIST"
]

if 'DATASET_X' not in globals() or len(DATASET_Y) == 0:
    print("[WARNING] Dataset is currently empty!")
    print("Please record samples in Cell 1.2 or click 'Load Synthetic Baseline' first.")
else:
    # Buat grid 2 baris x 4 kolom
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for idx, gesture_name in enumerate(GESTURE_ORDER):
        ax = axes[idx]
        
        # Cari indeks sampel untuk gestur ini
        sample_indices = [i for i, label in enumerate(DATASET_Y) if label == gesture_name]
        count = len(sample_indices)
        
        if count == 0:
            ax.text(0, 1.1, f"{gesture_name}\n(No Data Yet)", 
                    ha="center", va="center", color="gray", fontsize=11, fontweight="bold")
            ax.set_xlim(-1.6, 1.6)
            ax.set_ylim(-0.5, 3.0)
            ax.set_title(f"{gesture_name} (N = 0)", color="gray")
        else:
            sample_feats = np.array(DATASET_X[sample_indices[0]])
            pts = sample_feats.reshape(21, 2)
            
            # Koordinat x dan y (dibalik sumbu y agar jari mengarah ke atas)
            xs = pts[:, 0]
            ys = -pts[:, 1]
            
            # Draw finger bones (green lines)
            for p1, p2 in HAND_CONNECTIONS:
                ax.plot([xs[p1], xs[p2]], [ys[p1], ys[p2]], color="#27ae60", linewidth=2.5, zorder=2)
            
            # Draw joints (orange dots)
            ax.scatter(xs[1:20], ys[1:20], color="#e67e22", s=45, zorder=3)
            
            # Gambar ujung jari (titik kuning emas)
            tips = [4, 8, 12, 16, 20]
            ax.scatter(xs[tips], ys[tips], color="#f1c40f", s=75, edgecolors="#d35400", linewidth=1.5, zorder=4, label="Fingertips")
            
            # Draw wrist (red dot)
            ax.scatter(xs[0], ys[0], color="#e74c3c", s=90, edgecolors="#962d22", linewidth=1.5, zorder=5, label="Wrist (0,0)")
            
            ax.set_title(f"{gesture_name} (N = {count})", fontsize=12, fontweight="bold", pad=8)
            # Dynamic proportional axis limits so all hand joints and fingertips stay in view
            x_pad = max(float(np.abs(xs).max()), 1.2) + 0.35
            y_top = max(float(ys.max()) + 0.35, 2.8)
            y_bottom = min(float(ys.min()) - 0.25, -0.4)
            ax.set_xlim(-x_pad, x_pad)
            ax.set_ylim(y_bottom, y_top)
            ax.grid(True, linestyle="--", alpha=0.4)
        
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])

    plt.suptitle("MediaPipe 21-Joint Dataset Inspection (Stick Figures)", fontsize=16, fontweight="bold", y=0.98)
    plt.tight_layout()
    plt.show()
    print(f"Total dataset in memory: {len(DATASET_Y)} samples.")


## 1.3 Training the AI Brain: Random Forest Classifier

> **Quick Goal:** Teach the AI to read the 42-number finger joint tables and predict gesture names in milliseconds.

* **The 100 Junior Detectives Analogy (Random Forest):**
  * If we only ask a single friend to identify an object, they might make an honest mistake.
  * In a Random Forest, we recruit **100 junior detectives (Decision Trees)**.
  * Each detective checks 2–3 random clues (e.g., *"is the index finger extended?"*, *"is the thumb curled?"*).
  * At the end, all 100 detectives **cast a majority vote**. If 95 detectives shout "Thumbs Up!", the system is highly confident that the gesture is indeed a thumbs-up.
* **Why is this AI Architecture Ideal for Robotics?**
  * Training takes less than a second on a standard Colab CPU.
  * During live execution, it performs tens of predictions per second ($<2\text{ ms}$ inference) without requiring expensive GPU hardware.

### How It Works Behind the Scenes
* We split our dataset: 80% for learning (*Training Set*) and 20% for testing (*Test Set*).
* The model fits 100 voting trees, and the finalized model is serialized to `gesture_rf_model.joblib`.

> **Random Forest Ensemble Pipeline:**  
> `42 Joint Coordinates` ➔ `100 Decision Tree Detectives (Randomized Evaluation)` ➔ `Voting Ballot (Majority Vote)` ➔ `Final Output: Predicted Gesture`

> **Class Discussion Question:** *What would happen if we used only 1 decision tree instead of an ensemble? What risks arise when hands tremble or rotate slightly in the real world?*

In [ ]:
# --- DEFINE HYPERPARAMETERS HERE ---
n_trees = 100
max_tree_depth = 12
test_split_ratio = 0.20
random_seed = 42
# ---------------------------------------

import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

if len(DATASET_Y) < 16:
    print("[WARNING] Dataset is empty or has too few samples!")
    print("Please record samples in Cell 1.2 or click 'Load Synthetic Baseline'.")
else:
    X = np.array(DATASET_X)
    y = np.array(DATASET_Y)

    # Stratified Train and Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_split_ratio, random_state=random_seed, stratify=y
    )

    print(f"Training Random Forest on {len(X_train)} training samples...")
    rf_model = RandomForestClassifier(
        n_estimators=n_trees,
        max_depth=max_tree_depth,
        random_state=random_seed
    )
    rf_model.fit(X_train, y_train)

    # Evaluation on Test Data
    y_pred = rf_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"Model Accuracy on Test Data: {acc * 100:.2f}%\n")

    print("Classification Performance Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    # Confusion Matrix Visualization
    cm = confusion_matrix(y_test, y_pred, labels=rf_model.classes_)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf_model.classes_)
    disp.plot(ax=ax, cmap="Blues", colorbar=True, xticks_rotation=45)
    plt.title(f"Confusion Matrix Gesture Classifier (Accuracy: {acc*100:.1f}%)")
    plt.tight_layout()
    plt.show()

    # Save model for use in Chapter 3
    joblib.dump(rf_model, "gesture_rf_model.joblib")
    print("Model saved to 'gesture_rf_model.joblib'.")


### Reading the AI Report Card & The 100% Accuracy Pitfall

> **Quick Goal:** Interpret model accuracy using a Confusion Matrix and understand why a 100% score is not always completely bulletproof.

* **Reading the Evaluation Grid (Confusion Matrix):**
  * Numbers along the dark blue diagonal represent correct predictions.
  * Any numbers outside the diagonal indicate misclassifications (e.g., `PEACE` confused with `THREE_FINGERS`).
* **The 100% Trap (Overfitting):**
  * If your accuracy reaches 100%, do not celebrate prematurely!
  * If you only recorded samples with your hand locked in a single rigid pose, the AI merely memorized that exact photo rather than learning the generalized shape of the gesture.

## 1.4 Standalone Camera Testing (Offline / Without Connecting to Robot)

> **Quick Goal:** Test your live webcam to verify how reliably the AI recognizes your hand gestures before connecting to the physical robot arm.

* **Pre-Flight Test:**
  * Before operating an industrial machine weighing dozens of kilograms, we must verify our control signals safely in an offline environment.
  * Move your hand in front of your webcam and observe the predicted label box on the display.
* **The Flickering Phenomenon:**
  * Notice how predictions might flicker rapidly between classes when your fingers are mid-transition or slightly shaking.
  * **The Critical Bridge to Chapter 3:** This rapid flicker proves why we *never* directly wire raw camera output to robotic motors! In Chapter 3, we implement a **1-second Dwell Filter** to ensure the robot moves only when you hold an intentional, stable gesture.

> **Offline Verification Loop:**  
> `Laptop Webcam Frame` ➔ `Extract 21 Landmarks` ➔ `AI Prediction (<2 ms)` ➔ `Confidence Score Validation` ➔ `Live HUD Display`

> **Class Discussion Question:** *Try moving your hand very quickly across the camera frame. What happens to the predicted label? How can this issue be mitigated in a physical robotic control system?*

In [ ]:
#@title 1.4 Standalone Real-Time Camera Testing { display-mode: "form" }
# Please do not modify the contents of this cell
# 1.4 Prediction Functions & Real-Time Camera Testing (Offline / Without UR5)
import time
import asyncio
import cv2
import numpy as np
from IPython.display import HTML, Javascript, display
from google.colab.output import eval_js

def predict_gesture_with_confidence(feature_vector: np.ndarray):
    """Accepts 42 normalized features and returns predicted gesture name + confidence."""
    feats = feature_vector.reshape(1, -1)
    probs = rf_model.predict_proba(feats)[0]
    best_idx = np.argmax(probs)
    pred_label = rf_model.classes_[best_idx]
    confidence = float(probs[best_idx])
    return pred_label, confidence


OFFLINE_TESTER_JS = """
async function initOfflineTester() {
  const existing = document.getElementById('offline-box');
  if (existing) existing.remove();

  const container = document.createElement('div');
  container.id = 'offline-box';
  container.style.fontFamily = 'monospace';
  container.style.background = '#1a1a1a';
  container.style.color = '#fff';
  container.style.padding = '12px';
  container.style.borderRadius = '8px';
  container.style.display = 'inline-block';

  const header = document.createElement('div');
  header.innerHTML = '<b>Real-Time Gesture Model Testing (Offline / Without Robot)</b>';
  header.style.marginBottom = '6px';
  container.appendChild(header);

  const video = document.createElement('video');
  video.style.borderRadius = '6px';
  video.style.transform = 'scaleX(-1)';
  video.width = 400;
  video.height = 300;
  video.autoplay = true;
  video.muted = true;
  video.playsInline = true;
  container.appendChild(video);

  const hud = document.createElement('div');
  hud.id = 'offline-hud';
  hud.style.marginTop = '6px';
  hud.style.fontSize = '14px';
  hud.innerHTML = 'Status: Opening camera...';
  container.appendChild(hud);

  const btn = document.createElement('button');
  btn.innerText = 'Stop Testing';
  btn.style.marginTop = '8px';
  btn.style.background = '#e74c3c';
  btn.style.color = '#fff';
  btn.style.border = 'none';
  btn.style.padding = '6px 12px';
  btn.style.borderRadius = '4px';
  btn.style.cursor = 'pointer';

  btn.onclick = () => {
    window.offlineActive = false;
    if (window.offlineStream) window.offlineStream.getTracks().forEach(t => t.stop());
    btn.innerText = '⏹️ Testing Complete';
    btn.disabled = true;
    hud.innerHTML = 'Pengujian dinonaktifkan.';
  };
  container.appendChild(btn);

  document.querySelector('#output-body').appendChild(container);

  const stream = await navigator.mediaDevices.getUserMedia({video: {width: 400, height: 300}});
  window.offlineStream = stream;
  video.srcObject = stream;
  await video.play();

  const canvas = document.createElement('canvas');
  canvas.width = 400;
  canvas.height = 300;
  const ctx = canvas.getContext('2d');

  window.offlineActive = true;
  window.captureOfflineFrame = function() {
    if (!window.offlineActive) return null;
    ctx.drawImage(video, 0, 0, 400, 300);
    return canvas.toDataURL('image/jpeg', 0.6);
  };

  window.updateOfflineHUD = function(label, conf) {
    const el = document.getElementById('offline-hud');
    if (!el || !window.offlineActive) return;
    const pct = Math.round(conf * 100);
    const color = conf >= 0.7 ? '#2ecc71' : '#e67e22';
    el.innerHTML = `Prediksi: <b style="color:${color}; font-size:16px;">[${label}]</b> | Confidence: <b>${pct}%</b>`;
  };
}
initOfflineTester();
"""

async def test_gesture_model_offline(duration_seconds: float = 30.0):
    """Menjalankan visualisasi inferensi real-time di browser tanpa koneksi ke robot UR5."""
    display(Javascript(OFFLINE_TESTER_JS))
    await asyncio.sleep(2.0)

    print("Offline camera testing active! Point your hand gestures toward the camera.")
    print(f"Testing will run automatically for {duration_seconds} seconds (or click the Stop button).\\n")

    t_start = time.time()
    try:
        while (time.time() - t_start) < duration_seconds:
            is_active = eval_js('window.offlineActive ? true : false')
            if not is_active:
                break

            frame_b64 = eval_js('window.captureOfflineFrame ? window.captureOfflineFrame() : null')
            if frame_b64 is None:
                await asyncio.sleep(0.05)
                continue

            header, encoded = frame_b64.split(',', 1) if ',' in frame_b64 else ('', frame_b64)
            img_bytes = base64.b64decode(encoded)
            img_np = np.frombuffer(img_bytes, dtype=np.uint8)
            frame = cv2.imdecode(img_np, flags=cv2.IMREAD_COLOR)
            rgb = cv2.cvtColor(cv2.flip(frame, 1), cv2.COLOR_BGR2RGB)

            lms = hands_detector.detect_landmarks(rgb)
            if lms is not None:
                feats = extract_normalized_features(lms)
                if feats is not None:
                    label, conf = predict_gesture_with_confidence(feats)
                    eval_js(f'window.updateOfflineHUD ? window.updateOfflineHUD("{label}", {conf}) : null')
            else:
                eval_js('window.updateOfflineHUD ? window.updateOfflineHUD("NO HAND DETECTED", 0.0) : null')

            await asyncio.sleep(0.04)

    finally:
        eval_js('if (window.offlineStream) { window.offlineStream.getTracks().forEach(t => t.stop()); window.offlineActive = false; }')
        print("Offline testing complete.")


# Run offline camera testing (30 seconds)
await test_gesture_model_offline(duration_seconds=30.0)


# Chapter 2: Complex Robot Commands Using Large Language Models (LLM)

## 2.0 Conceptual Context: From Human Language to Physical Action

In Chapter 1, we built **perception**: Camera → Hand Landmarks → 42 Features → Gesture Label. In Chapter 3, these gesture labels map directly to waypoint commands. Chapter 2 bridges the gap in between: how **natural language sentences** are converted into structured action plans executable by a robot.

### Pitfalls of Raw Text Commands
If we forwarded raw human text directly to robot actuators:
* Sentences are **ambiguous** — "put it next to the box" contains no executable coordinates.
* There is no guaranteed **ordering** — the robot might close its gripper before descending to the table.
* There is no **state awareness** — the model does not know if the gripper is already holding an item or if the target location is occupied.

### Solution: LLM as a High-Level Planner, Not a Direct Controller
The LLM is strictly constrained to output **action symbols** from a closed set. Physical execution is governed by `robot_server.py`, which enforces safety guardrails, workspace limits, and mutex lease management. This decoupling ensures the robot remains safe even if the language model outputs an incorrect sequence.

| Layer | Responsibility | Location in Module |
| --- | --- | --- |
| Perception | Gesture / Sentence → Intent | Chapters 1 & 2.1 |
| Planning | Intent → Valid Action Sequence | Chapters 2.2–2.3 |
| Validation | Guardrails & Occupancy Rules | Chapter 2.3 (Simulation) & Server |
| Actuation | Action → Physical UR5 Motion | Chapter 2.4 |

> **Crucial Rule:** Models running in reasoning / thinking mode reject forced `tool_choice`. Do not use `tool_choice={"type": "function", ...}`. Send standard system/user messages instead.

> **Server Action Vocabulary (Ground Truth):** `move to home`, `box`, `pos1`–`pos4`, `pos1-up`–`pos4-up`, `open gripper`, `close gripper`, `task_completed`, `reject_task`.

## 2.1 Chatting with the LLM

### Why Start with a Basic Chat?
Before commanding the robot, we first verify that the connection to the LLM API is operational. A chat completion is the simplest contract: we send a **message list** with `system` (rules) and `user` (instructions) roles, and receive an `assistant` response.

In this cell, there are no tools, no robots, and no WebSockets. The sole objective is proving that the API key, base URL, and model are accessible from the Colab environment.

> **Specification:** Use `model="deepseek-flash"` and omit `tool_choice` parameters. Some reasoning models return HTTP 400 when tool choice is forced.

API keys are fetched from **Colab Secrets** under the key `DEEPSEEK_API_KEY`. Never hardcode raw API keys inside notebook cells.

### What Do We Learn from this Output?
Notice that `llm_history` appends two entries each time `llm_chat()` is called: one `user` message and one `assistant` message. This conversational history gives the interaction continuity—the LLM references previous exchanges on every subsequent call.

Try sending two consecutive prompts and observe how the second response builds upon the first. In Chapter 2.3, this conversational history evolves into an **action history** tracking physical task progress.

In [ ]:
!pip install -q openai

import asyncio
import json
import os
import re
import time
from openai import OpenAI

try:
    from google.colab import userdata as _userdata
except ImportError:
    _userdata = None

DEEPSEEK_BASE_URL = "https://api.deepseek.com"
MODEL_NAME = "deepseek-flash"
llm_history = []

def get_deepseek_key() -> str:
    key = ""
    if _userdata:
        try:
            key = (_userdata.get("DEEPSEEK_API_KEY") or "").strip()
        except Exception:
            pass
    return key or os.environ.get("DEEPSEEK_API_KEY", "").strip()

def llm_chat(user_text: str) -> str:
    key = get_deepseek_key()
    assert key, "Add DEEPSEEK_API_KEY in Colab Secrets (key icon in left sidebar)."
    llm_history.append({"role": "user", "content": user_text})
    client = OpenAI(api_key=key, base_url=DEEPSEEK_BASE_URL)
    resp = client.chat.completions.create(model=MODEL_NAME, messages=llm_history)
    answer = resp.choices[0].message.content or ""
    llm_history.append({"role": "assistant", "content": answer})
    return answer

print(llm_chat("Jelaskan dalam satu kalimat: apa bedanya obrolan dan perintah robot?"))
assert llm_history and llm_history[-1]["role"] == "assistant"
print("Total pesan dalam riwayat:", len(llm_history))


## 2.2 Context and Prompt Engineering

### The Problem: The LLM Cannot See the Table
A language model only sees the text we provide. It cannot see the camera feed, does not know if an item was moved by a previous student, and does not know if the gripper is empty. Without this state data, the LLM will **hallucinate** world conditions.

### The Solution: Prompt as a World State Snapshot
We inject the ground truth state into the system prompt on every decision cycle:

| Context Field | Meaning | Why It Is Mandatory |
| --- | --- | --- |
| `current_item_positions` | Location of each item (`box`, `pos1`–`pos4`, `gripper`) | The sole *ground truth*; prevents grabbing non-existent items |
| `holding_item` | Currently grasped item | Prevents the gripper from attempting double grasps |
| `waypoint` | Current arm position / elevation | Prevents gripping in mid-air (`-up`) |
| `history` | List of already executed actions | Prevents repeating already completed pick-and-place cycles |

Two rules that are commonly confused:
* **Reject** must only be invoked at the start (e.g., item already at destination, or destination position full).
* **Task completed** is called once the item has been safely placed—never call `reject_task` after a successful placement.

> You can modify `context` in the following cell to observe how changing a single field shifts the LLM's planned response.

In [ ]:
ALLOWED_ACTIONS = [
    "move to home", "box", "pos1", "pos2", "pos3", "pos4",
    "pos1-up", "pos2-up", "pos3-up", "pos4-up",
    "open gripper", "close gripper", "task_completed", "reject_task",
]

def build_system_prompt(ctx: dict) -> str:
    items = ", ".join(f"{k}={v}" for k, v in ctx["current_item_positions"].items())
    history = " | ".join(ctx.get("history", [])) or "Belum ada langkah."
    return "\n".join([
        "You are an action planner for a UR5 robot arm equipped with a Robotiq gripper.",
        "Respond with EXACTLY ONE action, chosen strictly from this list: " + ", ".join(ALLOWED_ACTIONS) + ".",
        "Current item positions (ground truth): " + items + ".",
        "Currently holding: " + str(ctx.get("holding_item") or "none") + ".",
        "Current waypoint: " + str(ctx.get("waypoint") or "home") + ".",
        "Action history: " + history + ".",
        "Rules: pos1..pos4 and box are table positions; posN-up are safe transit heights (gripping is prohibited there).",
        "If the item is already at its destination or the destination is full, respond with reject_task.",
        "Once all instructions are fulfilled, respond with task_completed. Do not respond with reject_task after an item has been placed.",
    ])

context = {
    "current_item_positions": {"item_1": "box", "item_2": "pos3"},
    "holding_item": None,
    "waypoint": None,
    "history": [],
}

prompt = build_system_prompt(context)
assert all(a in prompt for a in ALLOWED_ACTIONS)
assert "item_1=box" in prompt and "Riwayat aksi" in prompt
print(prompt)

def llm_plan(prompt_task: str, ctx: dict) -> str:
    key = get_deepseek_key()
    client = OpenAI(api_key=key, base_url=DEEPSEEK_BASE_URL)
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": build_system_prompt(ctx)},
            {"role": "user", "content": prompt_task},
        ],
    )
    return resp.choices[0].message.content or ""

print("\nContoh jawaban LLM:", llm_plan("Ambil item 1 lalu taruh di pos 2", context))


## 2.3 Action Planner Simulation

### Why Simulate Before Touching the Robot?
A single out-of-order command on physical hardware can damage the gripper or crash into the table. Therefore, planning logic must be validated against a software simulator before physical dispatch.

### What Does `ActionSimulator` Do?
* Maintains an in-memory representation of table state (`positions`, `holding`, `waypoint`).
* Validates every action against physical constraint rules.
* Rejects illegal actions with detailed error messages (e.g., closing gripper while holding an item, or dropping an item into an occupied slot).

```
[User Instruction] + [World Context]
            │
            ▼
     [LLM Reasoning]
            │
            ▼ (Proposed Action)
   [ActionSimulator]
      ├── Invalid? ──► Append error to prompt & retry (Self-Correction)
      └── Valid?   ──► Update state & proceed
```

In [ ]:
class ActionSimulator:
    TABLE = ("box", "pos1", "pos2", "pos3", "pos4")

    def __init__(self, positions: dict):
        self.positions = dict(positions)
        self.holding = None
        self.waypoint = None

    def snapshot(self):
        return {"current_item_positions": dict(self.positions),
                "holding_item": self.holding, "waypoint": self.waypoint}

    def apply(self, action: str):
        action = action.strip()
        if action in ("task_completed", "reject_task"):
            return True, action, self.snapshot()
        if action == "move to home":
            self.waypoint = None
            return True, action, self.snapshot()
        if action in self.TABLE or action.endswith("-up"):
            self.waypoint = action
            return True, action, self.snapshot()
        if action in ("open gripper", "close gripper"):
            if self.waypoint not in self.TABLE:
                return False, "Grip hanya diperbolehkan di posisi meja, bukan di waypoint -up.", self.snapshot()
            if action == "close gripper":
                if self.holding:
                    return False, "Gripper sudah membawa item.", self.snapshot()
                found = next((k for k, v in self.positions.items() if v == self.waypoint), None)
                if not found:
                    return False, "Tidak ada item di posisi ini.", self.snapshot()
                self.positions[found] = "gripper"
                self.holding = found
            else:
                if not self.holding:
                    return False, "Gripper kosong, tidak ada yang dilepas.", self.snapshot()
                if self.waypoint in self.positions.values():
                    return False, "Posisi tujuan sudah penuh.", self.snapshot()
                self.positions[self.holding] = self.waypoint
                self.holding = None
            return True, action, self.snapshot()
        return False, "Aksi tidak dikenal: " + action, self.snapshot()

# Uji aturan simulator tanpa LLM dan tanpa robot.
sim = ActionSimulator({"item_1": "box", "item_2": "pos3"})
ok, reason, _ = sim.apply("close gripper")
assert not ok, "Grip di udara harus ditolak"
ok, reason, _ = sim.apply("box")
assert ok
ok, reason, _ = sim.apply("close gripper")
assert ok and sim.holding == "item_1"
ok, reason, _ = sim.apply("pos2-up")
assert ok
ok, reason, _ = sim.apply("open gripper")
assert not ok, "Grip di -up harus ditolak"
ok, reason, _ = sim.apply("pos2")
ok, reason, _ = sim.apply("open gripper")
assert ok and sim.positions["item_1"] == "pos2"
print("Semua aturan simulator lolos. State akhir:", sim.snapshot())

def parse_action(text: str):
    for a in ALLOWED_ACTIONS:
        if re.search(r"\b" + re.escape(a) + r"\b", text, re.IGNORECASE):
            return a
    return None

def plan_and_simulate(task: str, sim: ActionSimulator, max_steps: int = 12):
    ctx = sim.snapshot()
    history = []
    for step in range(max_steps):
        raw = llm_plan(task, ctx)
        act = parse_action(raw)
        if not act:
            history.append("Step %d: jawaban tidak valid (%s)" % (step + 1, raw[:60]))
            ctx["feedback"] = "Pilih tepat satu aksi dari daftar yang diizinkan."
            continue
        ok, reason, snap = sim.apply(act)
        if not ok:
            history.append("Step %d: %s DITOLAK (%s)" % (step + 1, act, reason))
            ctx["feedback"] = reason
            ctx["history"] = history
            continue
        history.append("Step %d: %s" % (step + 1, act))
        ctx = snap
        ctx["history"] = history
        if act in ("task_completed", "reject_task"):
            break
    return sim, history

sim2 = ActionSimulator({"item_1": "box"})
sim2, riwayat = plan_and_simulate("Ambil item 1 lalu taruh di pos 1", sim2)
print("\n".join(riwayat))
print("State akhir:", sim2.snapshot())


## 2.4 Live Deployment to the Physical UR5 Robot

### From Simulator to Hardware Actuation
Now that the planner has been validated in simulation, we connect to the physical UR5 robot. The architecture mirrors the simulator, but each valid action is dispatched across WebSockets to `robot_server.py`.

### Safety Rules for Physical Execution
* **Emergency Stop (E-STOP):** The red E-STOP button in the notebook UI immediately aborts the server queue and halts the UR5.
* **Time-Leased Mutex:** Your group must acquire the control lease before the server accepts task execution commands.
* **Supervised Autonomous Cycle:** The planner runs step-by-step; observe each physical waypoint transition before the next action is triggered.

In [ ]:
# CELL 9 — Kirim Task & E-STOP ke UR5 (tanpa live log, tanpa loop)
# Jalankan cell sekali untuk menampilkan UI. Tombol tetap bisa dipakai
# selama kernel Colab hidup.

import asyncio
import json
import os
import time
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import userdata as _userdata
except ImportError:
    _userdata = None


def _api_key() -> str:
    key = ""
    if _userdata:
        try:
            key = (_userdata.get("DEEPSEEK_API_KEY") or "").strip()
        except Exception:
            pass
    return key or os.environ.get("DEEPSEEK_API_KEY", "").strip()


def _ws_url() -> str:
    url = ""
    if _userdata:
        try:
            url = (_userdata.get("TUNNEL_WS_URL") or "").strip()
        except Exception:
            pass
    return url or os.environ.get("TUNNEL_WS_URL", "").strip() or "wss://ur5.fawzqi.my.id"


def _fire(coro):
    # Colab memanggil callback widget saat loop kernel sedang berjalan,
    # jadi jangan buat loop baru — jadwalkan di loop yang sudah ada.
    task = asyncio.ensure_future(coro)
    task.add_done_callback(lambda t: t.exception() if not t.cancelled() else None)


async def _send(payload: dict, wait_for: str):
    import websockets
    async with websockets.connect(_ws_url(), open_timeout=8) as ws:
        await ws.send(json.dumps({
            "msg_id": f"m_{int(time.time()*1000) % 10**6}",
            "timestamp": time.time(),
            **payload,
        }))
        # Do not close connection before server completes: server sends
        # STEP_PROGRESS/PLAN_COMPLETE lewat koneksi ini. Kalau ditutup lebih
        # task on server fails if connection closes prematurely.
        while True:
            try:
                msg = json.loads(await asyncio.wait_for(ws.recv(), timeout=300))
            except asyncio.TimeoutError:
                break
            mt = msg.get("msg_type")
            p = msg.get("payload", {})
            if mt == wait_for or mt == "ERROR" or (mt == "PLAN_ACK" and p.get("accepted") is False):
                break


def _on_send(_):
    prompt = task_box.value.strip()
    if not prompt or not _api_key():
        return
    session = globals().get("session")
    token = getattr(session, "token", None) if session else None
    _fire(_send({
        "msg_type": "PROMPT_TASK",
        "payload": {"prompt": prompt, "api_key": _api_key(), "model": "deepseek-flash", "token": token},
    }, "PLAN_COMPLETE"))


def _on_estop(_):
    _fire(_send({
        "msg_type": "EMERGENCY_STOP",
        "payload": {"reason": "E-STOP dari Colab"},
    }, "EMERGENCY_STOP_ACK"))


task_box = widgets.Text(value="Pick item 1 and place it on pos 1", description="Task:",
                        layout=widgets.Layout(width="430px"))
send_btn = widgets.Button(description="Send Task", button_style="success", icon="paper-plane",
                          layout=widgets.Layout(width="130px"))
estop_btn = widgets.Button(description="E-STOP", button_style="danger", icon="stop",
                           layout=widgets.Layout(width="110px"))

send_btn.on_click(_on_send)
task_box.on_submit(_on_send)
estop_btn.on_click(_on_estop)

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;color:#89b4fa;'>🤖 UR5 Task Control</h3>"),
    widgets.HBox([task_box, send_btn, estop_btn], layout=widgets.Layout(margin="6px 0")),
]))


### Chapter 2 Summary

| Concept | Component | Function |
| --- | --- | --- |
| World State Injection | `build_system_prompt()` | Ground truth context preventing hallucination |
| Reactive Planning | `plan_and_simulate()` | Step-by-step action selection with self-correction |
| Safety Simulation | `ActionSimulator.apply()` | Rejection of mid-air gripping and slot overfilling |
| Physical Actuation | Cell 2.4 | `PROMPT_TASK` → `PLAN_COMPLETE` on lab server |

### Pre-Execution Checklist
1. The simulator in 2.3 passes all assertions.
2. The `robot_server.py` console displays `Received message: [PROMPT_TASK]` upon button press.
3. If the robot is in use by another group, wait for lease expiration (maximum 60–300s).
4. Test the **E-STOP** button once before executing lengthy tasks.
5. Understand the distinction between `REJECTED` (declined upfront) and `COMPLETED` (successfully placed).

# Chapter 3: Hands-on Learning with the UR5 Robotic Manipulator

In Chapter 3, we connect our programs to the real industrial UR5 robot arm in the laboratory. This module is built with modular flexibility, supporting both hand gesture teleoperation and automated action sequences.

---

## 3.1 Lab Network Architecture & Time-Leased Mutex Lock

> **Quick Goal:** Establish a secure communication channel from Google Colab to the lab workstation and understand collaborative queue management to prevent conflicting robot commands.

* **The Cloudflare Tunnel Bridge:**
  * Google Colab runs in Google's cloud infrastructure, while the physical UR5 robot sits on a laboratory workbench behind a local university firewall.
  * We utilize a secure **Cloudflare Tunnel** (WebSocket over SSL / WSS) as an encrypted virtual gateway, eliminating the need for public IP addresses or port forwarding.
* **The Restroom Key with an Egg Timer Analogy (Time-Leased Mutex Lock):**
  * There is only **1 physical robot arm** in the lab, but **dozens of students** across multiple groups operating laptops simultaneously.
  * If multiple participants trigger movements simultaneously, the robot would receive conflicting trajectories, trip safety limits, and trigger an emergency stop.
  * The lab server employs an automated digital lock: any group wishing to actuate the arm borrows a temporary control token (lease) for 30–60 seconds.
  * If your network connection drops while holding the lease, an internal timeout expires and automatically reclaims the token, preventing the robot from becoming permanently locked.

### How It Works Behind the Scenes
* The `RobotSession` client establishes a full-duplex asynchronous WebSocket connection.
* Any motion request is strictly rejected by the server unless accompanied by a valid, active lease token.

> **Communication & Leasing Pipeline:**  
> `Colab Notebook Client` ➔ `Cloudflare Tunnel (WSS)` ➔ `Lab Server Mutex Lock (Lease Timer)` ➔ `Physical UR5 Motion`

> **Class Discussion Question:** *Why must the control lease automatically expire upon inactivity rather than relying on manual release by the user?*

In [ ]:
#@title Robot Session Client Module (RobotSession) { display-mode: "form" }
# Please do not modify the contents of this cell
import asyncio
import json
import time
import websockets
def _msg(msg_type: str, payload: dict) -> str:
    """Standard envelope serializer helper for robot_server communication."""
    return json.dumps({
        "msg_id": str(int(time.time() * 1000) % 10**8),
        "msg_type": msg_type,
        "timestamp": time.time(),
        "payload": payload,
    })
class RobotSession:
    """
    Asynchronous communication client to Lab PC server.
    Manages mutex lock leasing lifecycle, quadrant sync, and robot motion execution.
    """
    def __init__(self, ws_url: str, student_id: str):
        self.ws_url = ws_url
        self.student_id = student_id
        self.token = None
        self.quadrant_config = None
    def set_quadrant_config(self, config: dict):
        """Saves active quadrant configuration to synchronize with lab server."""
        self.quadrant_config = dict(config)
        print(f"[SESSION] Quadrant configuration synchronized: {self.quadrant_config}")
    async def ping(self) -> dict:
        """Mengirim pesan heartbeat untuk menguji konektivitas server."""
        try:
            async with websockets.connect(self.ws_url, open_timeout=5.0) as ws:
                await ws.send(_msg("HEARTBEAT", {"client": self.student_id}))
                raw = await asyncio.wait_for(ws.recv(), timeout=5.0)
                return json.loads(raw).get("payload", {})
        except Exception as e:
            return {"error": str(e)}
    async def acquire_lock(self, duration_s: float = 45.0) -> bool:
        """Requests an exclusive control lease token for the robot."""
        try:
            async with websockets.connect(self.ws_url, open_timeout=5.0) as ws:
                req = _msg("ACQUIRE_LEASE", {"student_id": self.student_id, "duration_s": duration_s})
                await ws.send(req)
                raw = await asyncio.wait_for(ws.recv(), timeout=5.0)
                data = json.loads(raw).get("payload", {})
                if data.get("granted"):
                    self.token = data.get("token")
                    print(f"[LEASING] Control lease successfully acquired for '{self.student_id}'. Active for {duration_s} seconds.")
                    return True
                else:
                    holder = data.get("holder", "another group")
                    wait_time = data.get("wait_seconds", "?")
                    print(f"[LEASING] Robot is currently in use by '{holder}'. Remaining lease time: {wait_time} seconds.")
                    return False
        except Exception as e:
            print(f"[LEASING ERROR] Failed to reach lease manager server: {e}")
            return False
    async def release_lock(self) -> bool:
        """Releases control lease token so the robot can be used by other groups."""
        if not self.token:
            return True
        try:
            async with websockets.connect(self.ws_url, open_timeout=5.0) as ws:
                req = _msg("RELEASE_LEASE", {"student_id": self.student_id, "token": self.token})
                await ws.send(req)
                raw = await asyncio.wait_for(ws.recv(), timeout=5.0)
                data = json.loads(raw).get("payload", {})
                self.token = None
                print(f"[LEASING] Control lease for '{self.student_id}' safely released.")
                return data.get("released", True)
        except Exception as e:
            print(f"[LEASING ERROR] An issue occurred while releasing lease: {e}")
            self.token = None
            return False
    async def execute(self, action: str, quadrant_config: dict = None) -> bool:
        """Sends single action command to physical robot/simulator along with active quadrant config."""
        plan_id = f"plan_{int(time.time() * 1000) % 10**6}"
        cfg = quadrant_config or self.quadrant_config
        params = {"token": self.token, "client": self.student_id}
        if cfg:
            params["quadrant_config"] = cfg
        payload = {
            "plan_id": plan_id,
            "actions": [action],
            "parameters": params,
        }
        try:
            async with websockets.connect(self.ws_url, open_timeout=10.0) as ws:
                await ws.send(_msg("PLAN_DISPATCH", payload))
                ack_raw = await asyncio.wait_for(ws.recv(), timeout=8.0)
                ack = json.loads(ack_raw).get("payload", {})
                if not ack.get("accepted"):
                    print(f"[DISPATCH DITOLAK] {ack.get('message', 'Perintah tidak diterima oleh server')}")
                    return False
                print(f"[STATUS: RUNNING] Executing command [{action}]...")
                while True:
                    raw = await asyncio.wait_for(ws.recv(), timeout=35.0)
                    msg = json.loads(raw)
                    mtype = msg.get("msg_type")
                    p = msg.get("payload", {})
                    if mtype == "STEP_PROGRESS":
                        if p.get("status") == "SUCCESS":
                            elapsed = p.get("elapsed_s", 0.0)
                            print(f"[STATUS: SUCCESS] Completed in {elapsed:.2f} seconds.")
                        elif p.get("status") == "FAILED":
                            print(f"[STATUS: FAILED] Step execution failed: {p.get('message')}")
                            return False
                    elif mtype == "PLAN_COMPLETE":
                        status = p.get("status")
                        return status == "COMPLETED"
        except Exception as e:
            print(f"[EKSEKUSI ERROR] Koneksi terputus saat eksekusi [{action}]: {e}")
            return False
print("Modul klien RobotSession siap digunakan.")


## 3.2 Group Identity and Server URL Configuration

> **Quick Goal:** Set a unique group identifier so your control lease on the lab server is never confused with other participant teams.

* **Unique Group Name:** Replace `Group_01` with your team's designated ID (e.g., `Group_03`).
* **Why Must It Be Unique?** If two groups submit identical IDs, the server considers them the same client, causing race conditions for the active lease token!
* **Server Address:** Verify that the WebSocket tunnel address (`wss://ur5.fawzqi.my.id`) matches the instructor's projection screen.

### How It Works Behind the Scenes
* The Colab Form enables clean parameter input without modifying raw Python code. The instantiated `session` object manages network communication in memory.

In [ ]:
#@title Robot Control Session Configuration Form { display-mode: "form" }
# --- CONFIGURE GROUP IDENTITY HERE ---
STUDENT_ID = "Group_01"  #@param {type:"string"}
ROBOT_TUNNEL_WS = "wss://ur5.fawzqi.my.id"  #@param {type:"string"}
# -------------------------------------------
session = RobotSession(ws_url=ROBOT_TUNNEL_WS, student_id=STUDENT_ID)
print(f"Session client initialized for: {STUDENT_ID}")
print(f"Target Server: {ROBOT_TUNNEL_WS}")


## 3.3 Initial Handshake & HOME_POS Benchmark Test

> **Quick Goal:** Test network connectivity, acquire a temporary lease token, and command the robot to transit to the safe `HOME_POS` benchmark.

* **3 Safe Verification Steps:**
  * **Step 1 (Ping Heartbeat):** Verify that the network tunnel between Colab and the Lab PC is responsive.
  * **Step 2 (Acquire Lease):** Request an exclusive hardware control token from the server.
  * **Step 3 (Benchmark Motion):** Command the robot arm to transit to the pre-calibrated `HOME_POS` coordinate.
* **The `HOME_POS` Benchmark:** An upright, safe rest configuration verified by lab instructors to confirm that Colab commands successfully actuate physical joint motors.

### How It Works Behind the Scenes
* `verify_initial_connection()` sends a ping heartbeat, acquires a temporary lease, executes `HOME_POS`, and cleanly releases the token upon completion.

> **Handshake Verification Flow:**  
> `1. Ping Heartbeat` ➔ `2. Acquire Lease (30s)` ➔ `3. Transit to HOME_POS Benchmark` ➔ `4. Release Lease Cleanly`

> **Class Discussion Question:** *If the ping test succeeds but the lease request is denied, is the network connection broken? What is actually happening on the server?*

In [ ]:
#@title Execute Handshake Test & HOME_POS Benchmark { display-mode: "form" }
async def verify_initial_connection():
    print("[1/3] Sending ping signal to lab server...")
    status = await session.ping()
    if "error" in status:
        print(f"[WARNING] Failed to connect to server: {status['error']}")
        print("Pastikan server dan terowongan Cloudflare di lab PC dalam keadaan aktif.")
        return False
    print(f"[2/3] Successfully connected. Server lease status: {status.get('lease_status')}")
    
    print(f"[3/3] Requesting control lease for initial benchmark test...")
    has_lock = await session.acquire_lock(duration_s=30.0)
    if not has_lock:
        print("Lock is currently held by another group. Please try again in a few moments.")
        return False
    print("Mengirim perintah uji coba: titik benchmark [HOME_POS] dari captured_points.json...")
    ok_home = await session.execute("HOME_POS")
    if ok_home:
        print("Benchmark point [HOME_POS] executed successfully!")
    await session.release_lock()
    
    if ok_home:
        print("Chapter 3 initial test successful! Robot session, waypoint database, and network are ready.")
    return ok_home
await verify_initial_connection()


### Evaluation of Handshake Test 3.3: Three Verified Milestones

> **Quick Goal:** Confirm all foundational subsystems are functional before moving to advanced teleoperation.

* **1. Secure Network Tunnel Active:** Instruction packets successfully traversed the Cloudflare tunnel to the campus lab computer.
* **2. Concurrency Control Verified:** The digital mutex token safely prevented simultaneous multi-user hardware contention.
* **3. Hardware Actuation Confirmed:** The lab controller decoded waypoint coordinates and actuated the physical robot joints smoothly.

## 3.4 Chessboard Workspace Grid (A1 - H8) & Action Mapping

> **Quick Goal:** Discretize the robot workspace using an $8 \times 8$ chessboard coordinate system and map hand gestures to target quadrants.

* **The Chessboard & Battleship Analogy:**
  * Instead of telling the robot *"move 14.3 cm forward and 8.7 cm left"*, we place a standard $8 \times 8$ chessboard in front of the robot.
  * We simply command: *"Go to square B2!"* The robot controller references pre-calibrated Cartesian coordinates.
* **4 Table Quadrants:**
  * `Q1` (Front-Left) : Square `B2`
  * `Q2` (Front-Right) : Square `G2`
  * `Q3` (Back-Left) : Square `G7`
  * `Q4` (Back-Right) : Square `B7`
  * `BOX` (Placement Bin) : Calibrated drop-off container beside the workspace table.
* **Customizable Squares:** You may modify quadrant square assignments in the code (e.g., setting `Q1` to `C4`). The server validates chessboard bounds automatically.

> **Chessboard Grid & Quadrant Layout:**  
> `[Q3: Back-Left (G7)]` ── `[Q4: Back-Right (B7)]`  
> `[Q1: Front-Left (B2)]` ── `[Q2: Front-Right (G2)]` ➔ `[Drop-off Container: BOX]`

> **Class Discussion Question:** *Why is guiding a robot via discrete chessboard squares (B2, G2) significantly safer than unconstrained continuous fingertip tracing in 3D space?*

In [ ]:
#@title Chessboard Quadrant Configuration & Action Mapping { display-mode: "form" }
# --- CONFIGURE QUADRANTS HERE ---
# 3.4 Chessboard Quadrant Configuration & Gesture Action Mapping
# Students can customize target squares on the chessboard (A1 - H8) and the drop-off container BOX.
QUADRANT_CONFIG = {
    "Q1": "B2",   # Quadrant 1 (A1-D4)
    "Q2": "G2",   # Quadrant 2 (E1-H4)
    "Q3": "G7",   # Quadrant 3 (E5-H8)
    "Q4": "B7",   # Quadrant 4 (A5-D8)
    "BIN": "BOX", # Wadah peletakan barang
}
# Mapping from Machine Learning predictions to robot actions
GESTURE_TO_ACTION = {
    "POINT": "HOVER_Q1",         # 1 Finger -> Hover Quadrant 1
    "PEACE": "HOVER_Q2",         # 2 Fingers -> Hover Quadrant 2
    "THREE_FINGERS": "HOVER_Q3", # 3 Fingers -> Hover Quadrant 3
    "FOUR_FINGERS": "HOVER_Q4",  # 4 Fingers -> Hover Quadrant 4
    "OK_SIGN": "PICK_SEQUENCE",  # OK Sign -> Sekuens Pick (Hover -> Buka -> Turun -> Jepit -> Angkat)
    "OPEN_PALM": "RELEASE_BIN",  # Telapak Buka -> Sekuens Place (Ke BOX -> Turun -> Buka -> Angkat)
    "THUMBS_UP": "READY",        # Jempol -> Posisi netral / Home
    "FIST": None,                # Kepalan -> Hold / Netral (Safety Deadzone)
}
# ── Validate Quadrant Configuration ──
def validate_quadrant_config(config: dict):
    """Ensures each chessboard square is valid before dispatching plan to physical robot."""
    valid_files = set("ABCDEFGH")
    valid_ranks = set("12345678")
    valid_squares = {f"{f}{r}" for f in valid_files for r in valid_ranks} | {"BOX"}
    required_keys = ["Q1", "Q2", "Q3", "Q4", "BIN"]
    for k in required_keys:
        if k not in config:
            raise KeyError(f"Key '{k}' not found in QUADRANT_CONFIG! Must define Q1, Q2, Q3, Q4, and BIN.")
        
        sq = str(config[k]).strip().upper()
        if sq not in valid_squares:
            raise ValueError(
                f"Chessboard square '{sq}' in [{k}] is INVALID!\n"
                f"   Petak harus berada dalam rentang A1 - H8 (misal: 'B2', 'G7') atau 'BOX'."
            )
    # Sinkronkan konfigurasi ke objek session agar otomatis dikirim ke server lab
    if "session" in globals() and hasattr(session, "set_quadrant_config"):
        session.set_quadrant_config(config)
    print("Validation Successful! All target squares are valid:")
    for q_key, sq in config.items():
        print(f"   • {q_key:<4} -> Petak [{sq}]")
# Jalankan validasi
validate_quadrant_config(QUADRANT_CONFIG)


## 3.5 Dwell Debouncer Filter & Motion Idempotency

> **Quick Goal:** Filter natural hand tremors so the robot only moves when the user intentionally holds a stable, deliberate gesture.

* **The Human Hand Tremor Problem:**
  * Human hands are never 100% rigid; micro-tremors occur naturally every millisecond.
  * The camera streams at 30 FPS. If the robot reacted to every fractional-second fluctuation, the motors would shudder violently and trip emergency safety stops.
* **The "Hold for 1 Second" Rule (Dwell Time):**
  * A gesture must be held steadily for at least 1.0 full second before triggering actuation.
  * If a finger merely passes through a pose for 0.3 seconds during a transition, the system safely ignores it.
* **Single-Shot Latch-and-Hold Rule (Idempotency):**
  * Once the robot reaches Quadrant 1, continuing to show 1 finger does not repeatedly re-trigger motion to Quadrant 1. The robot remains stationary until a new, distinct gesture is detected.

> **Dwell Debouncer Pipeline:**  
> `Hand in Camera (With Tremor)` ➔ `Dwell Test (Held Steady ≥ 1.0s?)` ➔ `Dispatch 1 Motion Command` ➔ `Latch Lock (Prevent Repeated Triggers)`

> **Class Discussion Question:** *Have you ever observed building elevators? If you repeatedly press the elevator call button, does it arrive any faster? How does that compare to the idempotency principle used here?*

In [ ]:
#@title Initialize Dwell Debouncer Stability Filter { display-mode: "form" }
# Please do not modify the contents of this cell
import time
class TeleopDebouncer:
    """
    Gesture stability filter:
      - Dwell Time: Hand must hold gesture steadily for at least 1.0 second.
      - Latch-and-Hold: A single gesture triggers exactly 1 motion execution.
      - Cooldown: 1.0-second delay after motion completes to prevent re-triggering.
    """
    def __init__(self, dwell_time_s: float = 1.0, cooldown_s: float = 1.0):
        self.dwell_time = dwell_time_s
        self.cooldown_time = cooldown_s
        self.current_candidate = None
        self.candidate_start_time = 0.0
        self.latched_gesture = None
        self.last_trigger_time = 0.0
    def update(self, detected_gesture: str) -> bool:
        now = time.time()
        if not detected_gesture or detected_gesture in ("FIST", "UNKNOWN", "NONE"):
            self.current_candidate = None
            self.latched_gesture = None
            return False
        if detected_gesture == self.latched_gesture:
            return False
        if (now - self.last_trigger_time) < self.cooldown_time:
            return False
        if detected_gesture != self.current_candidate:
            self.current_candidate = detected_gesture
            self.candidate_start_time = now
            return False
        elapsed = now - self.candidate_start_time
        if elapsed >= self.dwell_time:
            self.last_trigger_time = now
            self.latched_gesture = detected_gesture
            self.current_candidate = None
            return True
        return False
    def get_progress(self) -> float:
        if not self.current_candidate or self.candidate_start_time == 0.0:
            return 0.0
        return min(1.0, (time.time() - self.candidate_start_time) / self.dwell_time)
    def arm_cooldown(self):
        self.last_trigger_time = time.time()
debouncer = TeleopDebouncer(dwell_time_s=1.0, cooldown_s=1.0)
print("Filter Dwell Debouncer siap.")


## 3.6 Deterministic Scenario Testing (Pick and Place)

> **Quick Goal:** Execute a full end-to-end sequence (gesture mapping → lease acquisition → arm trajectory → lease release) deterministically before enabling live interactive cameras.

* **End-to-End Test Without Live Cameras:**
  * We run an automated simulation sequence: `POINT` (Quadrant 1) ➔ `OK_SIGN` (Pick) ➔ `OPEN_PALM` (Release to Box) ➔ `THUMBS_UP` (Return Home).
  * Demonstrates that waypoint interpolation, gripper actuation, and mutex leasing execute seamlessly from start to finish.
* **Safe Clearance Trajectory (Pick, Place, and Lift-Up Clearance):**
  * Observe the physical robot in the lab: quadrant transits and bin movements use smooth Joint Space (`move_j`) paths, eliminating kinematic singularities.
  * During container placement (`BOX`), the robot follows a safe 4-phase sequence: transit hover (`HOVER_BIN`), linear descent (`RELEASE_BIN`), open gripper, and **immediately lift up to safe clearance height (`HOVER_BIN`)**. This vertical clearance ensures the gripper never collides with container rims during return transits to `READY`.

> **Class Discussion Question:** *Why must the robot arm vertically lift up past the container rim before translating laterally toward the READY position? What happens if it moves sideways while still inside the box?*

In [ ]:
#@title Execute Deterministic Pick and Place Scenario { display-mode: "form" }
async def handle_gesture_trigger(gesture_name: str):
    """Accepts verified gesture name, maps action, and dispatches to robot."""
    action = GESTURE_TO_ACTION.get(gesture_name)
    if not action:
        return False
    print(f"\n[GESTURE] Detected: [{gesture_name}] -> Triggering Action: [{action}]")
    if not session.token:
        has_lock = await session.acquire_lock(duration_s=45.0)
        if not has_lock:
            return False
    # Send action along with active QUADRANT_CONFIG to lab server
    success = await session.execute(action, quadrant_config=QUADRANT_CONFIG)
    debouncer.arm_cooldown()
    return success
print("Running full deterministic Pick and Place scenario...")
scenario_gestures = ["POINT", "OK_SIGN", "OPEN_PALM", "THUMBS_UP"]
for g in scenario_gestures:
    await handle_gesture_trigger(g)
    await asyncio.sleep(1.0)
await session.release_lock()
print("Scenario execution complete.")


## 3.7 Interactive Real-Time Camera Teleoperation (Hands-on Challenge)

> **Quick Goal:** Integrate everything learned into a unified closed loop: the camera tracks your hand, AI classifies the gesture, and the physical UR5 robot arm moves in real time!

* **The Workshop Climax:**
  * All subsystems now operate synchronously in a closed loop:
    1. Your laptop camera captures hand frames locally.
    2. MediaPipe extracts 21 finger joint landmarks.
    3. The Random Forest model predicts the gesture label.
    4. The Dwell Filter verifies a 1.0-second intentional hold.
    5. The verified command streams across the internet tunnel to actuate the UR5 arm in the lab!
* **Teleoperation Guidelines:**
  * Sit or stand upright with clear lighting on your hand.
  * Present distinct gestures directly to your webcam.
  * Watch the on-screen progress bar: once it fills completely (1.0s hold), the motion command dispatches immediately to the physical robot!

> **Full Teleoperation Loop:**  
> `Laptop Webcam` ➔ `21 MediaPipe Landmarks` ➔ `Random Forest AI` ➔ `1.0s Dwell Filter` ➔ `Cloudflare Tunnel` ➔ `UR5 Physical Arm Motion`

> **Class Discussion Question:** *When switching gestures from Quadrant 1 to Quadrant 2, watch what happens to the on-screen countdown bar in your browser!*

In [ ]:
#@title Real-Time Camera Teleoperation Interface { display-mode: "form" }
# Please do not modify the contents of this cell
# 3.7 Live Webcam Teleoperation Loop in Google Colab
import base64
import time
import asyncio
import cv2
import numpy as np
from IPython.display import HTML, Javascript, display
from google.colab.output import eval_js
# JavaScript Bridge for Streaming Browser Webcam to Google Colab
LIVE_WEBCAM_JS = """
async function initTeleopWebcam() {
  const existing = document.getElementById('teleop-webcam-box');
  if (existing) existing.remove();
  const container = document.createElement('div');
  container.id = 'teleop-webcam-box';
  container.style.fontFamily = 'monospace';
  container.style.background = '#1a1a1a';
  container.style.color = '#00ffcc';
  container.style.padding = '14px';
  container.style.borderRadius = '10px';
  container.style.display = 'inline-block';
  container.style.boxShadow = '0 4px 12px rgba(0,0,0,0.5)';
  const header = document.createElement('div');
  header.innerHTML = '<span style="font-size:16px; font-weight:bold;">UR5 Gesture Teleoperation Stream</span>';
  header.style.marginBottom = '10px';
  container.appendChild(header);
  const video = document.createElement('video');
  video.id = 'teleop-video';
  video.style.borderRadius = '8px';
  video.style.transform = 'scaleX(-1)';
  video.width = 480;
  video.height = 360;
  video.autoplay = true;
  video.muted = true;
  video.playsInline = true;
  container.appendChild(video);
  const hud = document.createElement('div');
  hud.id = 'teleop-hud';
  hud.style.marginTop = '8px';
  hud.style.fontSize = '13px';
  hud.innerHTML = 'Status: Meminta izin akses webcam...';
  container.appendChild(hud);
  const btnRow = document.createElement('div');
  btnRow.style.marginTop = '10px';
  const stopBtn = document.createElement('button');
  stopBtn.id = 'teleop-stop-btn';
  stopBtn.innerText = 'Stop Teleoperation Stream';
  stopBtn.style.background = '#e74c3c';
  stopBtn.style.color = '#fff';
  stopBtn.style.border = 'none';
  stopBtn.style.padding = '8px 16px';
  stopBtn.style.borderRadius = '5px';
  stopBtn.style.cursor = 'pointer';
  stopBtn.style.fontWeight = 'bold';
  stopBtn.onclick = () => {
    window.teleopStopped = true;
    window.teleopActive = false;
    if (window.teleopStream) {
      window.teleopStream.getTracks().forEach(track => track.stop());
    }
    stopBtn.innerText = '⏹️ Teleoperation Ended';
    stopBtn.disabled = true;
    stopBtn.style.background = '#7f8c8d';
    hud.innerHTML = 'Status: Session stopped.';
  };
  btnRow.appendChild(stopBtn);
  container.appendChild(btnRow);
  document.querySelector('#output-body').appendChild(container);
  window.teleopReady = false;
  window.teleopStopped = false;
  window.teleopActive = false;
  try {
    const stream = await navigator.mediaDevices.getUserMedia({video: {width: 480, height: 360}});
    window.teleopStream = stream;
    video.srcObject = stream;
    await video.play();
    const canvas = document.createElement('canvas');
    canvas.width = 480;
    canvas.height = 360;
    const ctx = canvas.getContext('2d');
    window.teleopActive = true;
    window.teleopReady = true;
    window.captureTeleopFrame = function() {
      if (!window.teleopActive || window.teleopStopped) return null;
      ctx.drawImage(video, 0, 0, 480, 360);
      return canvas.toDataURL('image/jpeg', 0.65);
    };
    window.updateHUD = function(gesture, conf, dwell, action) {
      const el = document.getElementById('teleop-hud');
      if (!el || window.teleopStopped) return;
      const pct = Math.round(conf * 100);
      const dwellPct = Math.round(dwell * 100);
      el.innerHTML = `Gesture: <b>${gesture}</b> (${pct}%) | Dwell: <b>${dwellPct}%</b> | Next Action: <b>${action || '-'}</b>`;
    };
    hud.innerHTML = 'Status: <span style="color:#2ecc71;">Camera Active & Ready!</span>';
  } catch (err) {
    hud.innerHTML = `<span style="color:#e74c3c;">Camera access denied/failed: ${err.message}</span>`;
    window.teleopReady = false;
    window.teleopStopped = true;
  }
}
initTeleopWebcam();
"""
def decode_base64_frame(b64_data: str) -> np.ndarray:
    if not b64_data:
        return None
    header, encoded = b64_data.split(',', 1) if ',' in b64_data else ('', b64_data)
    img_bytes = base64.b64decode(encoded)
    img_np = np.frombuffer(img_bytes, dtype=np.uint8)
    frame = cv2.imdecode(img_np, flags=cv2.IMREAD_COLOR)
    return cv2.flip(frame, 1) if frame is not None else None
async def start_live_gesture_teleop(duration_seconds: float = 180.0, conf_threshold: float = 0.50):
    """
    Loop teleoperasi real-time:
      Webcam -> MediaPipe -> ML Model -> Debounce Dwell Filter -> Physical UR5 Execution
    """
    # 1. Ensure exclusive robot control lease is acquired for teleoperation session
    if not session.token:
        await session.acquire_lock(duration_s=duration_seconds + 15.0)
    # 2. Initialize camera widget in browser
    display(Javascript(LIVE_WEBCAM_JS))
    print("Waiting for camera permission from browser (click 'Allow' if prompted)...")
    camera_ready = False
    for _ in range(30):
        ready = eval_js('window.teleopReady ? true : false')
        stopped = eval_js('window.teleopStopped ? true : false')
        if ready:
            camera_ready = True
            break
        if stopped:
            break
        await asyncio.sleep(0.5)
    if not camera_ready:
        print("Camera could not be accessed or permission was denied by browser.")
        print("Tip: Click the lock icon in the Colab browser URL bar -> allow Camera (Camera: Allow), then re-run this cell.")
        if session.token:
            await session.release_lock()
        return
    print("Live Teleoperation Active! Point your hand gestures toward the camera.")
    print("Tahan gestur selama 1.0 detik untuk memicu gerakan lengan UR5.")
    print("Click the [Stop Teleoperation Stream] button on camera widget to exit.\\n")
    start_time = time.time()
    try:
        while (time.time() - start_time) < duration_seconds:
            # Periksa apakah user mengklik tombol Stop
            user_stopped = eval_js('window.teleopStopped ? true : false')
            if user_stopped:
                print("⏹️ Teleoperation session terminated by user.")
                break
            # A. Ambil frame JPEG base64 dari browser
            frame_b64 = eval_js('window.captureTeleopFrame ? window.captureTeleopFrame() : null')
            if frame_b64 is None:
                await asyncio.sleep(0.05)
                continue
            # B. Dekode frame gambar
            frame = decode_base64_frame(frame_b64)
            if frame is None:
                await asyncio.sleep(0.05)
                continue
            # C. Deteksi landmark MediaPipe
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands_detector.process(rgb)
            current_gesture = "NONE"
            current_conf = 0.0
            dwell_progress = 0.0
            mapped_action = None
            if results.multi_hand_landmarks:
                hand_lms = results.multi_hand_landmarks[0]
                feats = extract_normalized_features(hand_lms)
                if feats is not None:
                    # D. Prediksi gesture via Random Forest
                    current_gesture, current_conf = predict_gesture_with_confidence(feats)
                    mapped_action = GESTURE_TO_ACTION.get(current_gesture)
                    # E. Filter kestabilan dwell
                    if current_conf >= conf_threshold:
                        is_triggered = debouncer.update(current_gesture)
                        dwell_progress = debouncer.get_progress()
                        if is_triggered and mapped_action:
                            print(f"\\nGESTURE TRIGGER TERVERIFIKASI: [{current_gesture}] ({current_conf*100:.1f}%) -> Eksekusi [{mapped_action}]!")
                            # Eksekusi aksi ke robot fisik
                            await handle_gesture_trigger(current_gesture)
            # F. Perbarui visual HUD di browser
            safe_act = mapped_action if mapped_action else "-"
            eval_js(f'window.updateHUD ? window.updateHUD("{current_gesture}", {current_conf}, {dwell_progress}, "{safe_act}") : null')
            await asyncio.sleep(0.04)  # ~25 FPS pacing loop
    except asyncio.CancelledError:
        print("Teleoperation task dibatalkan.")
    except Exception as e:
        print(f"Exception dalam teleop loop: {e}")
    finally:
        # Cleanup browser camera track
        eval_js('if (window.teleopStream) { window.teleopStream.getTracks().forEach(t => t.stop()); window.teleopActive = false; window.teleopStopped = true; }')
        if session.token:
            await session.release_lock()
        print("Teleoperation session finished. Robot lock safely released.")
# Run live camera teleoperation (default active duration: 180 seconds / 3 minutes)
await start_live_gesture_teleop(duration_seconds=180.0, conf_threshold=0.50)


## 3.8 Summary of Hands-on UR5 Teleoperation

> **Quick Goal:** Consolidate the 5 core engineering takeaways mastered during today's workshop.

* **1. Distributed Robotics Architecture:** Your laptop performs computer vision processing locally, while the lab workstation handles physical motor execution.
* **2. Shared Resource Management (Mutex Lease):** Safely sharing a single physical robot arm across dozens of simultaneous users without instruction collisions.
* **3. Intelligent Geometry Over Raw Pixels:** 21 landmark skeletons make our AI exceptionally lightweight, fast, and invariant to room lighting or background clutter.
* **4. Signal Conditioning (Dwell Debouncer):** Eliminates natural human hand tremors, ensuring smooth, deterministic, and safe robotic actuation.
* **5. Clearance Trajectory Safety (Orthogonal L-Shape):** Enforces vertical elevation before lateral transit, guaranteeing the arm clears all obstacles and container rims.

# Chapter 4: Interaction and Control of Quadruped Mobile Robots (RoboDog)

In Chapter 4, students explore natural interaction with a four-legged quadruped robot (*RoboDog*). This module leverages body pose tracking and dynamic gestures to control robot stance, orientation, and navigation in real environments.

---

## 4.1 Setup and RoboDog API Connection

Before starting interaction, we establish communication between Google Colab and the RoboDog application programming interface (API) across the lab network.

In [ ]:
# --- WRITE ROBODOG API CONNECTION CONFIGURATION HERE ---
# [STUDENT TEAM AREA: ROBODOG MODULE]

ROBODOG_API_URL = "http://lab-pc.local:8080/api/robodog"

def connect_robodog():
    """Initialize handshake with RoboDog controller."""
    print(f"Connecting to RoboDog controller at {ROBODOG_API_URL}...")
    # Write your team connection initialization logic here
    print("RoboDog connection standby.")
    return True

connect_robodog()


## 4.2 Pose- and Gesture-Based Interaction with RoboDog

In this section, students map full-body postures (such as raising both hands, crouching, or specific arm poses) into RoboDog stance modes:
* Upright Standing Pose $\to$ Robot Mode `STAND`
* Crouching / Low Stance $\to$ Robot Mode `CROUCH` / `SIT`
* Waving Gesture $\to$ Interactive Greeting Motion

In [ ]:
# --- IMPLEMENT POSE-TO-ROBODOG ACTION MAPPING HERE ---
# [STUDENT TEAM AREA: ROBODOG MODULE]

def map_human_pose_to_robodog(pose_landmarks):
    """
    Accepts body landmarks (e.g., MediaPipe Pose 33-point)
    and returns RoboDog posture/stance commands.
    """
    # Write body posture classification logic here
    return "STAND"

print("RoboDog pose mapping module ready for development.")


## 4.3 RoboDog Navigation and Locomotion Control

Dispatch coordinated linear velocity $(v_x, v_y)$ and angular velocity $(\omega_z)$ commands to the quadruped leg joint controllers.

In [ ]:
# --- IMPLEMENT ROBODOG NAVIGATION COMMAND DISPATCH HERE ---
# [STUDENT TEAM AREA: ROBODOG MODULE]

def dispatch_robodog_motion(vx: float, vy: float, omega: float, duration_s: float = 1.0):
    """Dispatches navigation velocity commands to RoboDog."""
    print(f"RoboDog moving: vx={vx:.2f} m/s, vy={vy:.2f} m/s, omega={omega:.2f} rad/s")
    # Write navigation motion API call here
    return True

# dispatch_robodog_motion(0.5, 0.0, 0.0)


## 4.4 Integrated Challenges and Capstone Scenarios

### Multi-Robot Collaboration Challenge:
Design an end-to-end scenario where **RoboDog** detects the presence of an object, navigates toward it, and signals the **UR5 Robotic Manipulator** to pick up the object and deposit it into the storage container.

---

## Human-Robot Interaction Workshop Final Checklist
- [x] Chapter 0: Environment dependencies and detection model setup.
- [x] Chapter 1: Pose-posture feature engineering and Machine Learning model.
- [ ] Chapter 2: High-level UR5 commands using LLMs.
- [x] Chapter 3: Manipulator waypoint teleoperation and safety control.
- [ ] Chapter 4: RoboDog interaction and locomotion response.